<a href="https://colab.research.google.com/github/shravan1808/DeepLearningProjects/blob/main/Project_06/Project_06_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RegularizedClassifier(nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(256,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(64,3)
    )

  def forward(self,X):
    X  = self.network(X)
    return X

In [ ]:
model = RegularizedClassifier()
optimizer = optim.Adam(
    model.parameters(),
    lr = 0.005,
    weight_decay = 1e-4
)

loss_fn = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=20,eta_min=1e-5)

In [ ]:
from torch.utils.data import Dataset,DataLoader,random_split
torch.manual_seed(42)
X = torch.rand(400,256)
y = torch.randint(0,3,size=(400,))

class CustomDataset(Dataset):
  def __init__(self,X,y) -> None:
    self.X = X
    self.y = y
  def __len__(self):
    return len(self.X)
  def __getitem__(self,idx):
    return self.X[idx],self.y[idx]

dataset = CustomDataset(X,y)
train_dataset,val_dataset = random_split(dataset,[320,80])

trainloader = DataLoader(train_dataset,batch_size=32,shuffle=True)
valloader = DataLoader(val_dataset,batch_size=16,shuffle=False)


In [ ]:
loss_history = []
best_loss = float("inf")
for epoch in range(20):
  epoch_loss = 0
  model.train()
  for X_batch,y_batch in trainloader:
    optimizer.zero_grad()
    output = model(X_batch)
    loss = loss_fn(output,y_batch)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    optimizer.step()
    epoch_loss+=loss.item()
  model.eval()
  with torch.no_grad():
    val_loss = 0
    for X,y in valloader:
      output = model(X)
      loss = loss_fn(output,y)
      val_loss+=loss.item()
    avg_val_loss=val_loss/len(valloader)
    if avg_val_loss < best_loss:
      best_loss = avg_val_loss
      checkpoint = {
          "model_state_dict":model.state_dict(),
          "epoch":epoch+1,
          "best_loss":best_loss
      }
  scheduler.step()
  current_lr = optimizer.param_groups[0]["lr"]
  avg_loss = epoch_loss/len(trainloader)
  loss_history.append(avg_loss)
  print(f"Epoch [{epoch+1}/20] - Training_Loss: {loss_history[epoch]:.4f} | Validation_Loss : {avg_val_loss:.6f}| LR: {current_lr:.6f}")

Epoch [1/20] - Training_Loss: 1.1841 | Validation_Loss : 1.103761| LR: 0.004969
Epoch [2/20] - Training_Loss: 0.9314 | Validation_Loss : 1.114472| LR: 0.004878
Epoch [3/20] - Training_Loss: 0.7025 | Validation_Loss : 1.238639| LR: 0.004728
Epoch [4/20] - Training_Loss: 0.5037 | Validation_Loss : 1.473958| LR: 0.004523
Epoch [5/20] - Training_Loss: 0.3042 | Validation_Loss : 1.723044| LR: 0.004269
Epoch [6/20] - Training_Loss: 0.2168 | Validation_Loss : 1.849022| LR: 0.003972
Epoch [7/20] - Training_Loss: 0.1584 | Validation_Loss : 1.993805| LR: 0.003638
Epoch [8/20] - Training_Loss: 0.1432 | Validation_Loss : 2.208774| LR: 0.003276
Epoch [9/20] - Training_Loss: 0.1146 | Validation_Loss : 2.410504| LR: 0.002895
Epoch [10/20] - Training_Loss: 0.1146 | Validation_Loss : 2.536441| LR: 0.002505
Epoch [11/20] - Training_Loss: 0.0609 | Validation_Loss : 2.635238| LR: 0.002115
Epoch [12/20] - Training_Loss: 0.0987 | Validation_Loss : 2.772997| LR: 0.001734
Epoch [13/20] - Training_Loss: 0.0627

In [ ]:
torch.save(checkpoint, "best_model.pt")